In [2]:
import numpy as np

d = np.load(r'C:\npy_output\keypoints\NIA_SL_SEN0001_REAL01_F.npy')
print(d.dtype, d.shape)

float32 (118, 137, 3)


In [5]:
import json

d = json.load(open(r'C:\npy_output\label_map.json', 'r', encoding = 'utf-8'))
print(len(d))

320


In [6]:
import numpy as np; d=np.load(r'C:\npy_output\keypoints_norm\NIA_SL_SEN0010_REAL01_F.npy'); print(d.shape)

(146, 137, 3)


In [7]:
import numpy as np; d=np.load(r'C:\npy_output\keypoints_norm\NIA_SL_SEN0001_REAL01_F.npy'); print(d.shape[0], '프레임 / 3.884초 =', round(d.shape[0]/3.884, 1), 'fps')

118 프레임 / 3.884초 = 30.4 fps


## 데이터 전처리

In [ ]:
"""
수어 데이터 전처리 스크립트
1. 키포인트 정규화 (pose 중심점 기준 상대좌표 + 스케일 정규화)
2. 라벨 인코딩 (텍스트 → 숫자 인덱스)
3. 데이터셋 통계 출력
"""

import os
import json
import numpy as np
import csv
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

# ============================================================
# 경로 설정
# ============================================================
NPY_INPUT_DIR = r"C:\npy_output\keypoints"
NPY_OUTPUT_DIR = r"C:\npy_output\keypoints_norm"
LABEL_CSV_PATH = r"C:\npy_output\labels.csv"
LABEL_ENCODED_PATH = r"C:\npy_output\labels_encoded.csv"
LABEL_MAP_PATH = r"C:\npy_output\label_map.json"

# ============================================================
# 키포인트 인덱스 매핑
# pose: 0~24 (25개)
# hand_left: 25~45 (21개)
# hand_right: 46~66 (21개)
# face: 67~136 (70개)
# ============================================================
POSE_START = 0
POSE_END = 25
HAND_LEFT_START = 25
HAND_LEFT_END = 46
HAND_RIGHT_START = 46
HAND_RIGHT_END = 67
FACE_START = 67
FACE_END = 137

# pose 기준점: 목(index 1)을 중심으로 사용
POSE_NECK_IDX = 1


def normalize_single_npy(args):
    """
    단일 npy 파일 정규화
    - pose 목(neck) 기준 상대좌표 변환
    - 어깨 너비 기준으로 스케일 정규화
    """
    input_path, output_path = args

    try:
        data = np.load(input_path)  # (frames, 137, 3)
        frames, num_kp, channels = data.shape

        # x, y 좌표와 confidence 분리
        coords = data[:, :, :2].copy()  # (frames, 137, 2)
        confidence = data[:, :, 2:3].copy()  # (frames, 137, 1)

        normalized_frames = []

        for f in range(frames):
            frame_coords = coords[f]  # (137, 2)
            frame_conf = confidence[f]  # (137, 1)

            # 목(neck) 좌표를 중심점으로
            neck = frame_coords[POSE_NECK_IDX]  # (2,)

            # confidence가 0이면 해당 포인트가 감지 안 된 거
            # 목이 감지 안 됐으면 해당 프레임은 그냥 0으로
            if frame_conf[POSE_NECK_IDX, 0] < 0.1:
                normalized_frames.append(np.zeros((num_kp, 3), dtype=np.float32))
                continue

            # 중심점 기준 상대좌표
            relative = frame_coords - neck  # (137, 2)

            # 스케일 정규화: 어깨 너비 기준
            # pose에서 왼쪽 어깨(index 2), 오른쪽 어깨(index 5)
            left_shoulder = frame_coords[2]
            right_shoulder = frame_coords[5]
            shoulder_width = np.linalg.norm(left_shoulder - right_shoulder)

            if shoulder_width < 1e-6:
                # 어깨 너비가 너무 작으면 (감지 실패 등) 스케일링 스킵
                scale = 1.0
            else:
                scale = shoulder_width

            relative = relative / scale

            # confidence가 낮은 포인트는 좌표를 0으로
            mask = (frame_conf[:, 0] < 0.1)
            relative[mask] = 0.0

            # 다시 합치기: (137, 3) = x_norm, y_norm, confidence
            frame_result = np.concatenate([relative, frame_conf], axis=1)
            normalized_frames.append(frame_result)

        result = np.stack(normalized_frames, axis=0).astype(np.float32)
        np.save(output_path, result)

        return {
            "file": os.path.basename(input_path),
            "frames": frames,
            "success": True,
        }

    except Exception as e:
        print(f"[ERROR] {input_path}: {e}")
        return {"file": os.path.basename(input_path), "success": False}


def build_label_encoding(label_csv_path, encoded_csv_path, label_map_path):
    """
    라벨 텍스트 → 숫자 인덱스 변환
    sentence_label 기준으로 고유 라벨 매핑 생성
    """
    rows = []
    with open(label_csv_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)

    # 고유 형태소 수집 (개별 형태소 단위)
    all_morphemes = set()
    for row in rows:
        detail = json.loads(row["morpheme_detail"])
        for m in detail:
            all_morphemes.add(m["label"])

    # 정렬 후 인덱스 매핑 (0: PAD, 1: UNK, 2~: 실제 라벨)
    sorted_morphemes = sorted(all_morphemes)
    label_map = {"<PAD>": 0, "<UNK>": 1}
    for i, morph in enumerate(sorted_morphemes):
        label_map[morph] = i + 2

    # label_map 저장
    with open(label_map_path, "w", encoding="utf-8") as f:
        json.dump(label_map, f, ensure_ascii=False, indent=2)

    # 인코딩된 CSV 생성
    encoded_rows = []
    for row in rows:
        detail = json.loads(row["morpheme_detail"])
        encoded_seq = [label_map.get(m["label"], 1) for m in detail]
        encoded_rows.append({
            "video_id": row["video_id"],
            "sentence_label": row["sentence_label"],
            "encoded_label": json.dumps(encoded_seq),
            "num_morphemes": len(encoded_seq),
        })

    with open(encoded_csv_path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(
            f, fieldnames=["video_id", "sentence_label", "encoded_label", "num_morphemes"]
        )
        writer.writeheader()
        writer.writerows(encoded_rows)

    print(f"\n[라벨 인코딩 완료]")
    print(f"  고유 형태소 수: {len(sorted_morphemes)}")
    print(f"  전체 클래스 수: {len(label_map)} (PAD, UNK 포함)")
    print(f"  라벨 맵: {label_map_path}")
    print(f"  인코딩 CSV: {encoded_csv_path}")

    # 상위 10개 라벨 출력
    from collections import Counter
    label_counter = Counter()
    for row in rows:
        detail = json.loads(row["morpheme_detail"])
        for m in detail:
            label_counter[m["label"]] += 1

    print(f"\n  [상위 10개 형태소]")
    for label, count in label_counter.most_common(10):
        print(f"    {label}: {count}회")

    return label_map


def print_dataset_stats(npy_dir):
    """
    데이터셋 통계 출력
    """
    files = [f for f in os.listdir(npy_dir) if f.endswith(".npy")]
    frame_counts = []
    for f in files:
        data = np.load(os.path.join(npy_dir, f))
        frame_counts.append(data.shape[0])

    print(f"\n[데이터셋 통계]")
    print(f"  총 영상 수: {len(files)}")
    print(f"  프레임 수 — 최소: {min(frame_counts)}, 최대: {max(frame_counts)}, "
          f"평균: {np.mean(frame_counts):.1f}, 중앙값: {np.median(frame_counts):.0f}")

    # 프레임 수 분포
    bins = [0, 30, 60, 90, 120, 150, 200, 300, 500, 1000]
    hist, _ = np.histogram(frame_counts, bins=bins)
    print(f"\n  [프레임 수 분포]")
    for i in range(len(hist)):
        bar = "█" * (hist[i] // 10) if hist[i] > 0 else ""
        print(f"    {bins[i]:>4}~{bins[i+1]:>4}: {hist[i]:>5}개 {bar}")


def main():
    print("=" * 60)
    print("수어 데이터 전처리 시작")
    print("=" * 60)

    # --------------------------------------------------------
    # 1단계: 키포인트 정규화
    # --------------------------------------------------------
    os.makedirs(NPY_OUTPUT_DIR, exist_ok=True)

    npy_files = sorted([f for f in os.listdir(NPY_INPUT_DIR) if f.endswith(".npy")])
    print(f"\n정규화 대상: {len(npy_files)}개 npy 파일")

    tasks = [
        (
            os.path.join(NPY_INPUT_DIR, f),
            os.path.join(NPY_OUTPUT_DIR, f),
        )
        for f in npy_files
    ]

    num_workers = max(1, cpu_count() - 2)
    print(f"워커 수: {num_workers}")

    results = []
    with Pool(num_workers) as pool:
        for result in tqdm(
            pool.imap_unordered(normalize_single_npy, tasks),
            total=len(tasks),
            desc="정규화 진행",
        ):
            results.append(result)

    success = sum(1 for r in results if r["success"])
    print(f"\n[정규화 완료] {success}/{len(npy_files)}개 성공")

    # --------------------------------------------------------
    # 2단계: 라벨 인코딩
    # --------------------------------------------------------
    print("\n" + "-" * 60)
    label_map = build_label_encoding(LABEL_CSV_PATH, LABEL_ENCODED_PATH, LABEL_MAP_PATH)

    # --------------------------------------------------------
    # 3단계: 데이터셋 통계
    # --------------------------------------------------------
    print("\n" + "-" * 60)
    print_dataset_stats(NPY_OUTPUT_DIR)

    print("\n" + "=" * 60)
    print("전처리 끝. 이제 모델 만들면 됨 ㅋㅋ")
    print(f"  정규화된 npy: {NPY_OUTPUT_DIR}")
    print(f"  인코딩 라벨: {LABEL_ENCODED_PATH}")
    print(f"  라벨 맵: {LABEL_MAP_PATH}")
    print("=" * 60)


if __name__ == "__main__":
    main()

## NPY 변환 

In [ ]:
import os
import numpy as np
from pathlib import Path
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

from util.csvBuild import build_label_csv
from util.parseMorpheme import parse_morpheme
from util.videoProcessing import process_single_video
from config import KEYPOINT_DIR, KEYPOINT_PARTS, TOTAL_KEYPOINTS, MORPHEME_DIR, OUTPUT_LABEL_CSV, ANGLE_FILTER, OUTPUT_NPY_DIR

def main():

    print("=" * 60)
    print("AIHUB 수어 데이터셋 → npy 변환 시작")
    print(f"  Keypoint 경로: {KEYPOINT_DIR}")
    print(f"  Morpheme 경로: {MORPHEME_DIR}")
    print(f"  출력 경로: {OUTPUT_NPY_DIR}")
    print(f"  앵글 필터: {ANGLE_FILTER} (Front)")
    print(f"  키포인트: {TOTAL_KEYPOINTS}개 (2D)")
    print("=" * 60)

    # 출력 디렉토리 생성
    os.makedirs(OUTPUT_NPY_DIR, exist_ok=True)

    # --------------------------------------------------------
    # 1단계: Front 앵글 폴더만 필터링
    # --------------------------------------------------------
    all_folders = sorted(os.listdir(KEYPOINT_DIR))
    front_folders = [
        f for f in all_folders
        if os.path.isdir(os.path.join(KEYPOINT_DIR, f)) and f.endswith(ANGLE_FILTER)
    ]
    print(f"\nFront 앵글 폴더 수: {len(front_folders)}개")

    if len(front_folders) == 0:
        print("[ERROR] Front 앵글 폴더가 없음. 경로 확인해라.")

    # --------------------------------------------------------
    # 2단계: 멀티프로세싱으로 npy 변환
    # --------------------------------------------------------
    tasks = [
        (os.path.join(KEYPOINT_DIR, f), f, OUTPUT_NPY_DIR)
        for f in front_folders
    ]

    num_workers = max(1, cpu_count() - 2)  # CPU 2개는 남겨둠
    print(f"워커 수: {num_workers}")
    print(f"\nnpy 변환 시작...")

    results = []
    with Pool(num_workers) as pool:
        for result in tqdm(
            pool.imap_unordered(process_single_video, tasks),
            total=len(tasks),
            desc="변환 진행",
        ):
            if result is not None:
                results.append(result)

    print(f"\n[DONE] npy 변환 완료: {len(results)}/{len(front_folders)}개 성공")

    # 통계
    if results:
        frame_counts = [r["num_frames"] for r in results]
        print(f"  프레임 수 — 최소: {min(frame_counts)}, "
                f"최대: {max(frame_counts)}, "
                f"평균: {np.mean(frame_counts):.1f}")

    # --------------------------------------------------------
    # 3단계: morpheme 라벨 CSV 생성
    # --------------------------------------------------------
    print(f"\nmorpheme 라벨 CSV 생성 중...")
    build_label_csv(MORPHEME_DIR, ANGLE_FILTER, OUTPUT_LABEL_CSV)

    print("\n" + "=" * 60)
    print("전부 끝남. 수고했다 ㅋㅋ")
    print(f"  npy 파일: {OUTPUT_NPY_DIR}")
    print(f"  라벨 CSV: {OUTPUT_LABEL_CSV}")
    print("=" * 60)

if __name__ == "__main__":
    main()

## LSTM 베이스라인

In [ ]:
"""
수어 인식 LSTM 베이스라인 — 클립 단위 형태소 분류
1. morpheme start/end로 클립 구간 추출
2. 클립을 LSTM에 넣어서 형태소 분류
3. train/val 분할 후 학습 및 평가
"""

import os
import json
import csv
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.model_selection import train_test_split
from collections import Counter

# ============================================================
# 경로 설정
# ============================================================
NPY_DIR = r"C:\npy_output\keypoints_norm"
LABEL_CSV = r"C:\npy_output\labels.csv"
LABEL_MAP_PATH = r"C:\npy_output\label_map.json"
MODEL_SAVE_PATH = r"C:\npy_output\lstm_baseline.pt"

# ============================================================
# 하이퍼파라미터
# ============================================================
FPS = 30
INPUT_DIM = 137 * 3       # 137 keypoints * (x, y, confidence)
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.3
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 50
PATIENCE = 7               # early stopping
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 데이터셋 준비: 클립 추출
# ============================================================
def extract_clips(npy_dir, label_csv, label_map, fps=30):
    """
    morpheme의 start/end 타임스탬프로 npy에서 클립 구간 추출
    Returns: list of (clip_array, label_index)
    """
    clips = []

    with open(label_csv, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    skipped = 0
    for row in rows:
        video_id = row["video_id"]
        npy_path = os.path.join(npy_dir, f"{video_id}.npy")

        if not os.path.exists(npy_path):
            skipped += 1
            continue

        data = np.load(npy_path)  # (frames, 137, 3)
        total_frames = data.shape[0]

        morpheme_detail = json.loads(row["morpheme_detail"])

        for morph in morpheme_detail:
            label_text = morph["label"]
            start_sec = morph["start"]
            end_sec = morph["end"]

            # 타임스탬프 → 프레임 번호
            start_frame = int(start_sec * fps)
            end_frame = int(end_sec * fps)

            # 범위 클리핑
            start_frame = max(0, start_frame)
            end_frame = min(total_frames, end_frame)

            if end_frame <= start_frame:
                continue

            # 클립 추출
            clip = data[start_frame:end_frame]  # (clip_frames, 137, 3)

            # flatten: (clip_frames, 137*3)
            clip_flat = clip.reshape(clip.shape[0], -1)

            # 라벨 인덱스
            label_idx = label_map.get(label_text, 1)  # 1 = UNK

            clips.append((clip_flat, label_idx))

    print(f"  총 클립 수: {len(clips)}")
    print(f"  스킵된 영상: {skipped}")

    return clips


# ============================================================
# PyTorch Dataset
# ============================================================
class SignClipDataset(Dataset):
    def __init__(self, clips):
        """
        clips: list of (np.array (frames, feat_dim), label_idx)
        """
        self.clips = clips

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip, label = self.clips[idx]
        return torch.FloatTensor(clip), torch.LongTensor([label])


def collate_fn(batch):
    """
    가변 길이 클립을 패딩해서 배치로 묶기
    """
    clips, labels = zip(*batch)
    lengths = torch.LongTensor([len(c) for c in clips])
    clips_padded = pad_sequence(clips, batch_first=True, padding_value=0.0)
    labels = torch.cat(labels)
    return clips_padded, labels, lengths


# ============================================================
# LSTM 모델
# ============================================================
class SignLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(dropout)
        # bidirectional이라 hidden_dim * 2
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        # pack_padded_sequence로 패딩 무시
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, (hidden, _) = self.lstm(packed)

        # bidirectional: 양방향 마지막 hidden state 합치기
        # hidden shape: (num_layers * 2, batch, hidden_dim)
        forward_last = hidden[-2]   # 정방향 마지막 레이어
        backward_last = hidden[-1]  # 역방향 마지막 레이어
        combined = torch.cat([forward_last, backward_last], dim=1)  # (batch, hidden*2)

        out = self.dropout(combined)
        out = self.fc(out)
        return out


# ============================================================
# 학습 루프
# ============================================================
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for clips, labels, lengths in dataloader:
        clips = clips.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(clips, lengths)
        loss = criterion(outputs, labels)
        loss.backward()

        # gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for clips, labels, lengths in dataloader:
            clips = clips.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(clips, lengths)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


# ============================================================
# 메인
# ============================================================
def main():
    print("=" * 60)
    print(f"수어 인식 LSTM 베이스라인")
    print(f"Device: {DEVICE}")
    print("=" * 60)

    # 라벨 맵 로드
    with open(LABEL_MAP_PATH, "r", encoding="utf-8") as f:
        label_map = json.load(f)
    num_classes = len(label_map)
    print(f"클래스 수: {num_classes}")

    # 클립 추출
    print("\n클립 추출 중...")
    clips = extract_clips(NPY_DIR, LABEL_CSV, label_map, FPS)

    if len(clips) == 0:
        print("[ERROR] 추출된 클립이 없음. 경로 확인해라.")
        return

    # 클래스 분포 확인
    label_counts = Counter([c[1] for c in clips])
    print(f"  고유 라벨 수 (실제 등장): {len(label_counts)}")

    # train/val 분할 (8:2)
    train_clips, val_clips = train_test_split(
        clips, test_size=0.2, random_state=42
    )
    print(f"\n  Train: {len(train_clips)} 클립")
    print(f"  Val:   {len(val_clips)} 클립")

    # DataLoader
    train_dataset = SignClipDataset(train_clips)
    val_dataset = SignClipDataset(val_clips)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=collate_fn, num_workers=0,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=0,
    )

    # 모델 생성
    model = SignLSTM(
        input_dim=INPUT_DIM,
        hidden_dim=HIDDEN_DIM,
        num_classes=num_classes,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n모델 파라미터 수: {total_params:,}")

    # 클래스 가중치 (불균형 대응)
    all_labels = [c[1] for c in train_clips]
    class_counts = Counter(all_labels)
    weights = torch.zeros(num_classes)
    for cls_idx, count in class_counts.items():
        weights[cls_idx] = 1.0 / count
    weights = weights / weights.sum() * num_classes
    weights = weights.to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=3, factor=0.5, verbose=True
    )

    # 학습
    print("\n" + "=" * 60)
    print("학습 시작")
    print(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9}")
    print("-" * 60)

    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)

        print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.1%} | {val_loss:>10.4f} | {val_acc:>8.1%}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"\nEarly stopping at epoch {epoch}")
                break

    # 최종 결과
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    print("\n" + "=" * 60)
    print(f"최종 Val Accuracy: {val_acc:.1%}")
    print(f"모델 저장: {MODEL_SAVE_PATH}")
    print("=" * 60)


if __name__ == "__main__":
    main()